# Quantum Feature Augmentation for Financial Market Prediction - AWS X State Street
### YQuantum 2026 

| Section | What You'll Learn |
|---|---|
| 1. Setup | Environment and imports |
| 2. Quantum Approach | Quick reference for key quantum ML vocabulary |
| 3. Data Generation | Generate the simulated regime-switching market data |
| 4. Classical Baseline | Polynomial feature expansion + Linear Regression — your bar to beat |
| 5. Quantum Feature Maps | Encode classical data into quantum states using rotation & entangling gates |
| 6. Hybrid Workflows | Variational circuits as trainable feature transformers |
| 8. Dimensionality Reduction | Dealing with overfitting after feature augmentation |
| 9. Evaluation | Compare all approaches with consistent metrics |
| 10. Real Stock Data | Task 2 of the challenge |
| 11. Further Reading | Additional quantum feature engineering methods for a broader landscape |

(This notebook aims to be used as a reference for guiding your implementation.)

---
## 1. Environment Setup
We use **Amazon Braket SDK** for all quantum circuits. The `LocalSimulator` runs everything on your machine. When you're ready to scale, you can swap in a managed simulator or QPU with a one-line change.

Note: We recommend keeping in consideration the noise and shot-based circuit evaluations, and not be totally reliant on analytically evaluated quantum circuits for your conclusions

Link to Amazon Braket Documentation: [amazon-braket-documentation](https://docs.aws.amazon.com/braket/latest/developerguide/what-is-braket.html)

Amazon Braket Examples GitHub Repository
[amazon-braket-examples](https://github.com/amazon-braket/amazon-braket-examples)

In [ ]:
# PROVIDED — install dependencies (run once)
# Uncomment the line below if you haven't installed these yet
# On Braket notebook instances, braket-sdk/numpy/matplotlib are pre-installed.
# Use %pip (not !pip) to ensure packages install into the active kernel environment.
#%pip install -q scipy scikit-learn pandas seaborn pennylane
#%pip install amazon-braket-sdk amazon-braket-pennylane-plugin 

In [ ]:
# Imports
    
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Classical Machine Learning
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split


# Quantum Machine Learning
import pennylane as qml
from pennylane import numpy as pnp

# Amazon Braket
from braket.circuits import Circuit, Observable, ResultType
from braket.devices import LocalSimulator

# Reproducibility
np.random.seed(42)

# Local state-vector simulator (exact, no shot noise)
SV_DEVICE = LocalSimulator("default")

# Setting up Amazon Braket Environment

# N_QUBITS = 4  # One qubit per feature (X1, X2, X3, X4)

# Local Simulator
# dev_braket_local = qml.device(
#     "braket.local.qubit",
#     wires=N_QUBITS,
#     backend="default",  # Braket's local state vector simulator
# )



# AWS Quantum Hardware
# dev_braket = qml.device(
#     "braket.aws.qubit",
#     device_arn="arn:aws:braket:us-east-1::device/qpu/ionq/Aria-1",
#     wires=N_QUBITS,
#     s3_destination_folder=("your-bucket", "your-prefix"),
#     shots=1000,
# )


## 2. Quantum Approach: Quick Reference

Here's the specific vocabulary we'll use throughout:

| Term | Meaning |
|---|---|
| **Feature map** | A quantum circuit that encodes a classical data point **x** into a quantum state \|φ(x)⟩. Different circuits create different encodings. |
| **Hilbert space** | The exponentially large state space of *n* qubits (dimension 2ⁿ). A 10-qubit system lives in a 1024-dimensional space. |
| **Variational circuit** | A parameterized quantum circuit where gate angles are *learned* during training, like weights in a neural network. |
| **Expectation value** | Measuring an observable (e.g., Pauli-Z) on a qubit gives a number in [-1, 1]. We use these as features. |

**Why might quantum features help for financial data?**

Financial signals are buried in noise, and the relationships between features can be highly non-linear (e.g., regime-dependent interactions). Quantum feature maps project data into an exponentially large Hilbert space where these non-linear relationships may become linearly separable — similar in spirit to the classical kernel trick, but accessing a fundamentally different class of feature spaces.

## 3. Data Generation - Simulated Regime Process

Refer the challenge spec: a two-regime process where Regime 1 (75% of the time) has one set of relationships and Regime 2 (25%) has a different set. 

In [ ]:
# Simulated Data Generation

# NOTE: Data prepocessing for standardization/bounding could be helpful

## 4. Classical Baseline

Before touching any quantum circuit, establish a strong classical baseline. The challenge requires linear regression as the base model. Classically, We augment features with polynomial expansions (degree 2) - this gives the model access to all pairwise interactions (like x₁·x₃).

**Classical baseline is your bar to rigorously compare against**, If quantum features don't improve on this, that's a valid (and publishable) negative result.

In [ ]:
# classical baseline with polynomial features
# This is a simple classical approach, feel free to extend this with classically researched methods for equally elevated comparisons

# NOTE: Example Code
# X_train_s, y_train etc are undefined in the example below

# def evaluate_model(y_true, y_pred, label=""):
#     """Compute standard regression metrics."""
#     mse  = mean_squared_error(y_true, y_pred)
#     mae  = mean_absolute_error(y_true, y_pred)
#     corr = pearsonr(y_true, y_pred)[0]
#     return {"model": label, "MSE": round(mse, 6), "MAE": round(mae, 6), "Corr": round(corr, 4)}


# # --- Degree-1 (raw features) ---
# lr_raw = LinearRegression().fit(X_train_s, y_train)
# pred_raw = lr_raw.predict(X_test_s)
# results = [evaluate_model(y_test, pred_raw, "LR (raw features)")]

# # --- Degree-2 (polynomial features) ---
# poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
# X_train_poly = poly.fit_transform(X_train_s)
# X_test_poly  = poly.transform(X_test_s)

# lr_poly = LinearRegression().fit(X_train_poly, y_train)
# pred_poly = lr_poly.predict(X_test_poly)
# results.append(evaluate_model(y_test, pred_poly, "LR (poly deg=2)"))

# # --- Ridge with poly (regularized) ---
# ridge_poly = Ridge(alpha=1.0).fit(X_train_poly, y_train)
# pred_ridge = ridge_poly.predict(X_test_poly)
# results.append(evaluate_model(y_test, pred_ridge, "Ridge (poly deg=2)"))

# baseline_df = pd.DataFrame(results)
# print("=== Classical Baselines (Out-of-Sample) ===")
# baseline_df

## 5. Quantum Feature Maps

A **quantum feature map** is a circuit that transforms a classical vector **x** into a quantum state |φ(**x**)⟩. The key idea: by measuring this state, we extract new features that live in a space classical polynomial expansions cannot efficiently reach.

We cover three encoding strategies below, but you are by no means restricted in your implementation.

Pennylane documentation for encoding menthods as template: [pennylane-docs-template](https://docs.pennylane.ai/en/stable/introduction/templates.html)

#### 5.1 Angle Encoding (+ Entanglement Layer)
- **What it does**: Maps each feature xᵢ to a rotation angle on qubit i: Ry(xᵢ)\|0⟩ 
- **Key idea**: Each qubit independently encodes one feature. Entanglement would capture cross feature interactions.
 This is the quantum equivalent of "just feeding raw features in."
- **Qubits needed**: One per feature (n_features qubits), using more qubits compared to features is also valid stratergy
- **Paper**:  Schuld & Petruccione, *Supervised Learning with Quantum Computers* (Springer, 2018) 

In [ ]:
# Angle Encoding feature map
# Using Pennylane integration with Amazon Braket for your implementation of your quantum circuits could be a helpful tool

N_QUBITS = 4
N_FEATURES = 4  # X1, X2, X3, X4

dev = qml.device("default.qubit", wires=N_QUBITS) # local simulator 

@qml.qnode(dev)
def circuit_angle_basic_entangle(x, weights):
    """
    AngleEmbedding -> BasicEntanglerLayers -> Measurements.
    
    Args:
        x: input features, shape (4,)
        weights: trainable params for entangling layers, shape (n_layers, n_qubits)
    """
    # Encode classical data into qubit rotations
    qml.AngleEmbedding(x, wires=range(N_QUBITS), rotation="Y")
    
    # Entangling layers with trainable single-qubit rotations + CNOT ring
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    
    # Measure single-qubit and pairwise expectations
    single = [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    pairs = [
        qml.expval(qml.PauliZ(i) @ qml.PauliZ(j))
        for i in range(N_QUBITS) for j in range(i + 1, N_QUBITS)
    ]
    return single + pairs


#### 5.2 ZZ Feature Map

- **What it does**: Encodes features using rotations *and* entangling ZZ interactions between qubit pairs, creating cross-feature correlations in the quantum state.
- **Key idea**: After single-qubit rotations (Hadamard + Rz), pairs of qubits interact through ZZ(xᵢ·xⱼ) gates — implementing exp(i·xᵢ·xⱼ·ZZ). The circuit repeats for multiple layers, increasing expressiveness. This is the feature map with proven classical hardness of simulation in certain regimes.
- **Paper**: Havlíček et al., "Supervised learning with quantum-enhanced feature spaces," *Nature* 567, 209–212 (2019)

#### 5.3 IQP (Instantaneous Quantum Polynomial) Encoding

- **What it does**: A diagonal circuit (all gates commute) that applies Hadamards, encodes feature products via diagonal phase gates, then Hadamards again.
- **Key idea**: The resulting output distribution is believed to be classically hard to sample from, suggesting the feature space is genuinely beyond the reach of classical polynomial kernels. Same depth as ZZ map but a different algebraic structure.
- **Paper**: Shepherd & Bremner, "Temporally Unstructured Quantum Computation," *Proc. Royal Society A* (2009); applied to ML in Havlíček et al. (2019)

Pennylane IQP: [link](https://docs.pennylane.ai/en/stable/code/api/pennylane.IQPEmbedding.html)

In [ ]:
# TODO — Experiment with different feature maps and parameters
#
# Ideas to try (following is just for ideas and not a requirement):
# 1. Change the number of reps in zz_feature_map (reps=1, 2, 3)
# 2. Use iqp_encoding instead of zz_feature_map
# 3. Try 'probability' extraction instead of 'expectation'
# 4. Use varying number of qubits (e.g., 3, 10, 15) and see how it affects performance
# 5. Try Combine quantum features with polynomial classical features


## 6. Hybrid Quantum-Classical Workflows

In the previous sections, quantum circuits were fixed - no trainable parameters. In hybrid workflows we add **variational parameters**: gate angles optimized via classical gradient descent, just like weights in a neural network. 

The intended use for the requirement of utilizing a linear regression model would be trainining your hybrid tranformer to produce augmented features that are produced by the circuit according to a valid/relevant objective. Varitional Quantum Circuits can also result in being their own autonomous predictors, feel free to experiment with it after meeting the regression model requirement for comparisons as in the handout.

#### 6.1 Variational Quantum Circuit (VQC) as Feature Transformer

- **What it does**: A parameterized circuit where some gate angles encode input data and others are *trainable*. The circuit's expectation values serve as a learned feature transformation.
- **Key idea**: Alternating data-encoding layers and variational layers let the circuit *adapt* its feature extraction to the task. Think of it as a quantum neural network layer that learns which features to pull out of the Hilbert space.
- **Paper**: Mitarai et al., "Quantum Circuit Learning," *PRA* 98, 032309 (2018)

In [ ]:
# VQC training loop (parameter-shift gradient estimation)

# Phase 1: Train the VQC to produce good features (minimize a proxy loss)
# Phase 2: Extract features with trained circuit -> fit regression model

# The proxy loss in this formulation can be:
#   (a) Direct MSE on circuit outputs (treats each ⟨Z_i⟩ as a feature)
#   (b) Kernel-target alignment (unsupervised circuit optimization)
#   (c) Feature variance maximization

#### 6.2 Quantum Reservoir Computing

- **What it does**: Uses a *random*, untrained quantum circuit as a fixed nonlinear feature extractor. Only the classical readout layer is trained.
- **Key idea**: Analogous to classical echo-state networks - the quantum circuit provides a rich, high-dimensional nonlinear transformation without needing gradient-based optimization. Different random seeds give different feature spaces; you can ensemble multiple reservoirs cheaply.
- **Paper**: Fujii & Nakajima, "Harnessing Disordered-Ensemble Quantum Dynamics for Machine Learning," *PRA* 96, 042306 (2017)

In [ ]:
# Experiment with hybrid workflows
#
# Ideas to try:
# 1. Train VQC with more epochs and larger training set
# 2. Change VQC architecture: more layers, different entangling patterns
# 3. Vary reservoir parameters: n_layers, n_reservoirs, different seeds
# 4. Combine reservoir features with quantum kernel (use reservoir features as
#    input to a classical RBF kernel)
# NOTE: These are to guide your ideas, and not restrain your implementation. We are looking for rigorous comparisons, you can be creative with your feature engineering approach as you like

### Extracting Features from Quantum States

Once data is encoded into a quantum state, we need to *extract* quantum-augmented features from it. Two main approaches:

1. **Expectation values**: Measure observables (Pauli-Z, X, Y or products) on each qubit. Each gives a real number in [-1, 1]. With *n* qubits you get *n* features per observable type.

2. **Probabilities**: Measure the probability of each computational basis state. With *n* qubits this gives 2ⁿ features (exponential — potentially richer but expensive). n qubits will result in 2ⁿ features, which could lead to overfitting in the regression model




## 8. Dimensionality Reduction
To deal with overfitting after augmented features, you can also employ dimensionality reduction tools within your model/circuit (Regularization).

## 9. Evaluation
Collect results from your implemented methods and visualize side-by-side. Remember: the challenge values **rigorous out-of-sample evaluation** over raw performance numbers.


## 10. Real World Scenario - Adapting to Real Stock Data

The second task requires you to work with **actual S&P 500 data**. Refer the challenge handout

NOTE: Make sure there is no data leaks between training and testing data when implementing Walk-forward backtesting.

---
## 11.  Further Reading - Additional Quantum Methods [Experimental]

The methods below are for the purpose of completion of the challenge statement. You are not required to implement these methods as they are for your understanding of the broader landscape.


#### Quantum Random Kitchen Sinks

- **What it does**: Approximates a quantum kernel via random features — analogous to classical random Fourier features for the RBF kernel - without building the full N×N kernel matrix.
- **Key idea**: Instead of computing all O(N²) kernel entries, generate a fixed set of random quantum feature vectors. Their inner products approximate the true quantum kernel. Scales linearly with dataset size rather than quadratically. The reservoir computing approach above is closely related.
- **Composed of**: Random parameterized circuits → Measurement → Classical feature vector → Linear model
- **When to use**: When the dataset is too large for full kernel matrix computation but you want the expressiveness of a quantum kernel.
- **Paper**: Wilson et al., "Quantum Kitchen Sinks: An Algorithm for Machine Learning on Near-Term Quantum Computers," arXiv:1806.08321 (2018)



#### QAOA-Inspired Feature Maps

- **What it does**: Borrows the QAOA circuit structure — alternating problem and mixer Hamiltonians — as a feature map where the problem Hamiltonian encodes the data.
- **Key idea**: QAOA circuits carry an inductive bias toward combinatorial structure. Encoding financial features into the problem Hamiltonian gives a feature map with a fundamentally different algebraic form than ZZ or IQP maps, potentially capturing different correlations.
- **Composed of**: Alternating layers exp(−iγ·C(x)) → exp(−iβ·B), where C(x) is a cost Hamiltonian parameterized by the data
- **When to use**: Experimental — worth trying if ZZ/IQP maps show no improvement.
- **Paper**: Farhi, Goldstone & Gutmann, "A Quantum Approximate Optimization Algorithm," arXiv:1411.4028 (2014)


#### Quantum Convolutional Neural Network (QCNN)

- **What it does**: Applies the CNN paradigm to quantum circuits — local parameterized gates act on neighboring qubits (convolution), followed by qubit measurement and discard (pooling) to progressively reduce dimensionality.
- **Key idea**: Each convolutional layer captures local qubit correlations; pooling layers reduce active qubits and suppress barren plateaus. The hierarchical structure mirrors classical CNNs and makes the circuit more noise-resilient.
- **Composed of**: Convolutional layer (2-qubit parameterized gates) → Pooling (measure and discard half the qubits) → Repeat → Readout
- **When to use**: When features have sequential or spatial structure — e.g., ordered time-series lags, price features at different lookback horizons.
- **Paper**: Cong, Choi & Lukin, "Quantum convolutional neural networks," *Nature Physics* 15, 1273–1278 (2019)


#### Quantum Transfer Learning

- **What it does**: Uses a pre-trained classical neural network for initial feature extraction, then routes the compressed representation into a variational quantum circuit for final processing.
- **Key idea**: Classical networks handle high-dimensional, noisy input well. The quantum circuit then operates on a manageable feature count (matching qubit count), potentially capturing correlations the classical network misses. Practically: train a classical autoencoder to reduce raw features to ~5–10 dimensions, then feed those into a VQC.
- **Composed of**: Classical NN (frozen weights) → Dimensionality reduction layer → VQC → Readout
- **When to use**: Experimental approach, useful when raw features far exceed the qubits available.
- **Paper**: Mari et al., "Transfer learning in hybrid classical-quantum neural networks," *Quantum* 4, 340 (2020)

---
**Good luck! Remember: rigorous methodology and honest conclusions matter more than beating the baseline.**